# 03 - Batches and Pairwise Operations

## Imports

In [1]:
import numpy as np

# Make sure we get outputs from a probabilitistic model each time (for reproducibility)
np.random.seed(1515)

## What's Missing From Notebook 02

`02-matrices-and-transformations.ipynb` covered the matrix-vector product for exactly *one* vector at a time ($Mv$), and composing two transformation matrices together. But every training loop built since - `deep_learning_primer/01`'s `W1 @ X`, `genai_architecture_primer/02`'s `Q @ K.T`, `unsupervised_learning_primer/01`'s distance-to-every-centroid computation - operates on a whole *batch* of vectors at once, and several of them compute *every pairwise relationship* between two batches in a single matrix multiplication. Nothing about the underlying math is new; it's the same matrix-vector product and the same dot product from `01` and `02`, just applied to many vectors simultaneously instead of one at a time. This notebook names that pattern explicitly, since so much downstream code depends on it silently.

## A Matrix as a Batch of Vectors

Instead of one input vector, stack many input vectors together as the *columns* of a matrix - exactly the `(features, N)` shape used throughout `deep_learning_primer`, `rl_primer`, and `genai_architecture_primer`. Multiplying that matrix by a transformation matrix $W$ applies the *exact same* transformation to every column at once.

In [2]:
# the 90-degree-ish rotation matrix from 02-matrices-and-transformations.ipynb
theta = np.radians(30)
R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

# 4 points, stacked as columns of one matrix - a "batch" of 4 vectors
points = np.array([
    [1.0, 0.0, 2.0, -1.0],
    [0.0, 1.0, 1.0,  0.5],
])

batched_result = R @ points   # rotate all 4 points in one matrix multiplication
looped_result = np.column_stack([R @ points[:, i] for i in range(points.shape[1])])

print("Batched result:\n", batched_result.round(3))
print("\nMatches doing it one vector at a time in a loop:", np.allclose(batched_result, looped_result))

Batched result:
 [[ 0.866 -0.5    1.232 -1.116]
 [ 0.5    0.866  1.866 -0.067]]

Matches doing it one vector at a time in a loop: True


That equivalence is the entire idea: `R @ points` and rotating each point one at a time produce identical results, but the batched version is one matrix multiplication instead of a Python loop. This is exactly what `deep_learning_primer/01`'s `W1 @ X` was doing every time it ran - transforming an entire batch of training examples in one shot, not looping over them.

## All-Pairs Operations: Comparing Every Vector to Every Other Vector

A different, equally common pattern: given two batches of vectors, compute *every* pairwise relationship between them at once. Recall the dot product from `01-vectors.ipynb` measures how much two vectors point in the same direction. If $A$ is a batch of vectors (as *rows* this time) and $B$ is another batch, then $A B^T$ computes the dot product of *every* row of $A$ against *every* row of $B$, all in one matrix multiplication.

In [3]:
A = np.random.randn(3, 4)   # 3 vectors, 4-dim each
B = np.random.randn(5, 4)   # 5 vectors, 4-dim each

all_pairs_dot = A @ B.T   # every row of A dotted with every row of B, at once
looped_dot = np.array([
    [np.dot(A[i], B[j]) for j in range(B.shape[0])]
    for i in range(A.shape[0])
])

print(f"Shape of A @ B.T: {all_pairs_dot.shape}  (one row per vector in A, one column per vector in B)")
print("Matches computing every pair in a double loop:", np.allclose(all_pairs_dot, looped_dot))

Shape of A @ B.T: (3, 5)  (one row per vector in A, one column per vector in B)
Matches computing every pair in a double loop: True


This exact operation - `A @ B.T` computing every pairwise dot product in one shot - is precisely `genai_architecture_primer/02-attention-mechanism.ipynb`'s `Q @ K.T`: every token's Query dotted with every token's Key, all at once, producing the whole grid of attention scores in a single matrix multiplication instead of a nested loop over token pairs.

## All-Pairs Distances via Broadcasting

A close cousin of the same idea, using *subtraction* instead of a dot product: computing the distance from every point in one batch to every point in another - exactly what `unsupervised_learning_primer/01-clustering.ipynb`'s K-Means needed (distance from every data point to every centroid) and what `perception_primer/05-objects-in-motion.ipynb`'s Hungarian Algorithm section needed (cost between every predicted box and every detected box). NumPy's broadcasting - inserting an extra axis so every combination lines up automatically - makes this a single expression instead of a loop, the same trick both of those notebooks already used without pausing to explain it.

In [4]:
X = np.random.randn(4, 2)   # 4 points
C = np.random.randn(3, 2)   # 3 reference points (e.g. centroids)

# X[:, None, :] has shape (4, 1, 2); C[None, :, :] has shape (1, 3, 2) - broadcasting
# lines every one of the 4 points up against every one of the 3 reference points
broadcast_distances = np.sum((X[:, None, :] - C[None, :, :]) ** 2, axis=2)
looped_distances = np.array([
    [np.sum((X[i] - C[j]) ** 2) for j in range(C.shape[0])]
    for i in range(X.shape[0])
])

print(f"Shape of broadcast_distances: {broadcast_distances.shape}  (4 points x 3 references)")
print("Matches computing every distance in a double loop:", np.allclose(broadcast_distances, looped_distances))

Shape of broadcast_distances: (4, 3)  (4 points x 3 references)
Matches computing every distance in a double loop: True


Same result, same shape, as a double loop computing each distance one at a time - just without ever writing the loop. This is exactly the line that computed distances in K-Means' assignment step and the Hungarian Algorithm's cost matrix; both were already using this pattern, just without a name attached to it.

## Try It Yourself

`unsupervised_learning_primer/01-clustering.ipynb`'s `kmeans` function computed distances with `np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)` instead of the `np.sum((...) ** 2, axis=2)` used above. Confirm these give the *same* cluster assignments, but not the same raw numbers - compute both versions on the same `X` and `C` from above, and explain in one sentence why one is the square of the other, and why that difference never actually changes which reference point ends up closest to any given point.

In [5]:
# TODO: compute both np.linalg.norm(...) and np.sum((...) ** 2, ...) on X and C above,
# compare the actual numbers, and confirm argmin along axis=1 gives the same result
# for both (i.e. the same nearest-reference-point assignment either way)


## Resources

- [NumPy: Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) - the official rules behind the `X[:, None, :] - C[None, :, :]` pattern used above (reference).
- [3Blue1Brown: Essence of Linear Algebra - Matrix Multiplication](https://www.3blue1brown.com/topics/linear-algebra) - revisit with the batched/pairwise framing in mind (video).